# 03 - Sentence-Transformer Fine-Tuning

Run this after notebook 01. It fine-tunes a sentence-transformer on resume/JD pairs using `match_score` as the similarity target.

In [ ]:
!pip -q install sentence-transformers

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

In [ ]:
DATA_PATH_CANDIDATES = [
    'cleaned_resumeJD_pairs.csv',
    '/content/cleaned_resumeJD_pairs.csv',
    '/content/drive/MyDrive/cleaned_resumeJD_pairs.csv',
]

data_path = next((p for p in DATA_PATH_CANDIDATES if os.path.exists(p)), None)
if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = next(iter(uploaded.keys()))
    except Exception as exc:
        raise FileNotFoundError('Upload cleaned_resumeJD_pairs.csv from notebook 01.') from exc

df = pd.read_csv(data_path)
df = df.dropna(subset=['resume_text', 'job_description', 'match_score', 'match_label']).copy()
df['match_score'] = pd.to_numeric(df['match_score'], errors='coerce')
df = df.dropna(subset=['match_score'])

print(f'Loaded: {len(df)} pairs')
print('\nLabel distribution:')
print(df['match_label'].value_counts())
print(f'\nScore range: {df["match_score"].min():.2f} - {df["match_score"].max():.2f}')
display(df.head(3))

In [ ]:
# Stratified train/val/test split so each split keeps high/medium/low examples.
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df['match_label']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['match_label']
)

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name}: {len(split)} rows | {split["match_label"].value_counts().to_dict()}')

In [ ]:
def make_examples(frame):
    return [
        InputExample(
            texts=[str(row['resume_text']), str(row['job_description'])],
            label=float(row['match_score'])
        )
        for _, row in frame.iterrows()
    ]

train_examples = make_examples(train_df)
val_examples = make_examples(val_df)
test_examples = make_examples(test_df)

print(f'Train examples: {len(train_examples)}')
print(f'Val examples:   {len(val_examples)}')
print(f'Test examples:  {len(test_examples)}')

In [ ]:
BASE_MODEL = 'all-mpnet-base-v2'
OUTPUT_DIR = 'finetuned_resume_jd_model'

model = SentenceTransformer(BASE_MODEL)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
train_loss = losses.CosineSimilarityLoss(model=model)

evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(val_examples, name='val')

print(f'Base model: {BASE_MODEL}')

In [ ]:
# Increase EPOCHS if you want stronger training. Keep it low for a quick Colab run.
EPOCHS = 1
warmup_steps = max(1, int(len(train_dataloader) * EPOCHS * 0.1))

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path=OUTPUT_DIR,
    show_progress_bar=True
)

print(f'Saved fine-tuned model to: {OUTPUT_DIR}')

In [ ]:
def predict_similarity(sentence_model, frame, batch_size=16):
    resume_emb = sentence_model.encode(frame['resume_text'].astype(str).tolist(), batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True)
    jd_emb = sentence_model.encode(frame['job_description'].astype(str).tolist(), batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True)
    return np.sum(resume_emb * jd_emb, axis=1)

base_model = SentenceTransformer(BASE_MODEL)
finetuned_model = SentenceTransformer(OUTPUT_DIR)

test_eval = test_df.copy()
test_eval['base_similarity'] = predict_similarity(base_model, test_eval)
test_eval['finetuned_similarity'] = predict_similarity(finetuned_model, test_eval)

def summarize(name, y_true, y_pred):
    return {
        'model': name,
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': mean_squared_error(y_true, y_pred) ** 0.5,
        'correlation': np.corrcoef(y_true, y_pred)[0, 1],
    }

metrics = pd.DataFrame([
    summarize('base', test_eval['match_score'], test_eval['base_similarity']),
    summarize('finetuned', test_eval['match_score'], test_eval['finetuned_similarity']),
])

display(metrics.round(4))
display(test_eval[['match_label', 'match_score', 'base_similarity', 'finetuned_similarity']].head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(test_eval['match_score'], test_eval['base_similarity'], alpha=0.65, label='Base')
axes[0].scatter(test_eval['match_score'], test_eval['finetuned_similarity'], alpha=0.65, label='Fine-tuned')
axes[0].plot([0, 1], [0, 1], 'r--')
axes[0].set_xlabel('True match_score')
axes[0].set_ylabel('Predicted similarity')
axes[0].set_title('Base vs Fine-tuned Similarity')
axes[0].legend()

plot_df = test_eval.melt(
    id_vars=['match_label'],
    value_vars=['base_similarity', 'finetuned_similarity'],
    var_name='model',
    value_name='similarity'
)
sns.boxplot(data=plot_df, x='match_label', y='similarity', hue='model', order=['low', 'medium', 'high'], ax=axes[1])
axes[1].set_title('Similarity by Label')

plt.tight_layout()
plt.show()

In [ ]:
test_eval.to_csv('finetune_test_predictions.csv', index=False)

metadata = {
    'base_model': BASE_MODEL,
    'dataset': 'cleaned_resumeJD_pairs.csv',
    'total_pairs': int(len(df)),
    'train_pairs': int(len(train_df)),
    'val_pairs': int(len(val_df)),
    'test_pairs': int(len(test_df)),
    'epochs': EPOCHS,
    'label_distribution': df['match_label'].value_counts().to_dict(),
    'metrics': metrics.to_dict(orient='records'),
}

with open('finetune_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved: finetune_test_predictions.csv')
print('Saved: finetune_metadata.json')
print(f'Saved model folder: {OUTPUT_DIR}')

try:
    from google.colab import files
    files.download('finetune_test_predictions.csv')
    files.download('finetune_metadata.json')
except Exception:
    pass